# Módulo 03: CRISP-DM, fases 1 y 2

En este notebook recorremos las dos primeras fases de CRISP-DM con el caso de la **inmobiliaria**:

1. **Comprensión del negocio:** ¿qué problema resolvemos y cómo sabremos que lo logramos?
2. **Comprensión de los datos:** ¿qué tenemos, qué tan bueno es y qué hipótesis nos sugiere?

> Basado en `sesion_2_crisp_dm.pdf` (profesor Oscar N. Gómez).

---
## Fase 1: Comprensión del negocio

**Contexto:** la inmobiliaria tarda en promedio 5 días en hacer un avalúo, porque un perito visita la casa. Los clientes se van con la competencia, que responde el mismo día.

| Nivel | Definición |
|---|---|
| **Problema de negocio** | Dar un precio de referencia en minutos, no en días |
| **Objetivo de ciencia de datos** | Modelo de **regresión** que estime `SalePrice` a partir de las características de la casa |
| **Criterio de éxito (negocio)** | El 80 % de los avalúos automáticos quedan a menos de ±15 % del avalúo del perito |
| **Criterio de éxito (modelo)** | MAPE < 15 % en datos que el modelo no ha visto, y mejor que el *baseline* |
| **Riesgos** | Casas atípicas (mansiones, remates), datos de 2006-2010 (¿siguen vigentes?), barrios con pocas ventas |

> El criterio de éxito **se define ANTES de modelar**. Si no, cualquier resultado "parece bueno".

---
## Fase 2: Comprensión de los datos

### 2.1 Cargar y mirar

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from intro_analitica.datos import cargar_house_prices, leer_diccionario_house_prices
from intro_analitica.eda import perfil_rapido

sns.set_theme(style="whitegrid")
df = cargar_house_prices()

print(f"Filas: {df.shape[0]:,} · Columnas: {df.shape[1]}")
df.head()

**¿Qué representa una fila?** Una **venta** de una casa. Es la primera pregunta que hay que responder siempre:
de ella depende cómo se agrupa, qué es un duplicado y qué se puede predecir.

### 2.2 El diccionario de datos

Con 81 columnas no podemos adivinar qué es `BsmtFinType2`. El archivo `data_description.txt` lo explica.
La función `leer_diccionario_house_prices()` lo convierte en un diccionario de Python.

In [ ]:
diccionario = leer_diccionario_house_prices()
for columna in ["SalePrice", "OverallQual", "GrLivArea", "PoolQC", "Alley", "BsmtFinType2"]:
    print(f"{columna:<14} → {diccionario.get(columna, '(no está en el diccionario)')}")

`SalePrice` no aparece en el diccionario porque es el **target** (la competencia de Kaggle lo documenta aparte).

Y mira lo que dice el diccionario sobre los códigos de `PoolQC`:

In [ ]:
from intro_analitica import config

texto = config.HOUSE_PRICES_DESCRIPCION.read_text(encoding="utf-8")
inicio = texto.find("PoolQC")
print(texto[inicio : inicio + 160])

**`NA = No Pool`.** Esto será clave en el módulo 04: muchos "nulos" **no son datos perdidos**, significan "no tiene".

### 2.3 Tipos de variables

In [ ]:
tipos = df.dtypes.astype(str).value_counts()
print(tipos)

numericas = df.select_dtypes("number").columns.drop(["Id", "SalePrice"])
categoricas = df.select_dtypes(exclude="number").columns
print(f"\nNuméricas: {len(numericas)} · Categóricas: {len(categoricas)}")

**Ojo:** que pandas diga "número" no significa que la variable sea **cuantitativa**:
- `MSSubClass` (tipo de vivienda) usa códigos 20, 30, 60… pero es **categórica**: 60 no es "el triple" de 20.
- `OverallQual` (1-10) es **ordinal**: hay orden, pero la distancia entre 1 y 2 no es necesariamente igual a la que hay entre 9 y 10.

In [ ]:
df[["MSSubClass", "OverallQual"]].nunique()

### 2.4 Perfil rápido: nulos y cardinalidad

`perfil_rapido` (de `src/intro_analitica/eda.py`) responde en una tabla: tipo, nulos, % de nulos, valores únicos y un ejemplo.

In [ ]:
perfil = perfil_rapido(df)
perfil[perfil["nulos"] > 0]

In [ ]:
con_nulos = perfil[perfil["nulos"] > 0]["pct_nulos"].sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
con_nulos.plot(kind="barh", ax=ax, color="indianred")
ax.set(title="% de nulos por columna", xlabel="% de filas")
plt.show()

**Lectura:** 19 columnas tienen nulos. Hay tres grupos:

| Grupo | Columnas | ¿Qué significa probablemente? |
|---|---|---|
| Casi todo nulo (>80 %) | `PoolQC`, `MiscFeature`, `Alley`, `Fence` | La casa **no tiene** esa característica |
| Familias de columnas | `Garage*` (81), `Bsmt*` (37-38) | Casas sin garaje o sin sótano: los nulos aparecen **juntos** |
| Pocos nulos sueltos | `Electrical` (1), `MasVnrArea` (8) | Datos realmente perdidos |

In [ ]:
# ¿Los nulos de garaje aparecen juntos? Si sí, es la misma casa "sin garaje".
columnas_garaje = ["GarageType", "GarageFinish", "GarageQual", "GarageCond", "GarageYrBlt"]
print("Filas con algún nulo de garaje :", df[columnas_garaje].isna().any(axis=1).sum())
print("Filas con TODOS nulos de garaje:", df[columnas_garaje].isna().all(axis=1).sum())
print("De esas, ¿cuántas tienen GarageArea = 0?:", (df.loc[df[columnas_garaje].isna().all(axis=1), "GarageArea"] == 0).sum())

Confirmado: las 81 casas con nulos de garaje tienen `GarageArea = 0`. **No tienen garaje.**

### 2.5 La variable objetivo (target)

In [ ]:
print(df["SalePrice"].describe().apply("{:,.0f}".format))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(df["SalePrice"], bins=50, ax=axes[0])
axes[0].set(title="SalePrice", xlabel="USD")
sns.boxplot(x=df["SalePrice"], ax=axes[1])
axes[1].set(title="Boxplot de SalePrice", xlabel="USD")
plt.tight_layout()
plt.show()

El precio tiene **asimetría a la derecha** (cola larga de casas caras). Lo tendremos en cuenta al modelar: el error en las casas caras pesa mucho en el RMSE.

### 2.6 Primeras relaciones (solo para generar hipótesis)

In [ ]:
correlaciones = df[numericas.tolist() + ["SalePrice"]].corr()["SalePrice"].drop("SalePrice").sort_values()
fig, ax = plt.subplots(figsize=(6, 4))
pd.concat([correlaciones.head(4), correlaciones.tail(8)]).plot(kind="barh", ax=ax, color="teal")
ax.set(title="Correlación con SalePrice (extremos)", xlabel="Pearson")
plt.show()

### 2.7 Problemas detectados e hipótesis (entregable de la fase 2)

**Problemas de calidad**
1. 19 columnas con nulos; la mayoría significa "no tiene" (piscina, garaje, sótano…).
2. Variables numéricas que en realidad son categóricas (`MSSubClass`).
3. Target asimétrico, con casas muy caras.
4. Los datos son de 2006-2010: ¿siguen representando el mercado actual? (**riesgo de vigencia**).

**Hipótesis**
- H1: la **calidad general** (`OverallQual`) es el factor que más explica el precio (correlación ≈ 0.79).
- H2: a mayor **área habitable** (`GrLivArea`), mayor precio, pero con *outliers* (casas enormes y baratas).
- H3: el **barrio** influye aun cuando se compara por calidad y área.
- H4: las casas **remodeladas** recientemente se venden más caras que otras de la misma edad.

Estas hipótesis se validan en el **módulo 05 (EDA)**.

---
## Actividad puente al módulo 04: ¿qué harías con estos datos?

Del material del profesor: una tabla pequeña pero **sucia**. En parejas, identifiquen **todos** los problemas antes de mirar la solución.

In [ ]:
ventas_sucias = pd.DataFrame(
    {
        "ID": [1, 2, 3, 4],
        "Fecha": ["2023/12/05", "12-06-23", "2023/12/07", "2023/12/05"],
        "Precio": [59.99, None, -15, 89.90],
        "Cliente": ["Juan P.", "Maria G", None, "Carlos S."],
        "Ciudad": ["Bogotá", "Bta", "Medellin", "bogotá"],
    }
)
ventas_sucias

<details>
<summary>Ver los problemas (¡primero piénsalo!)</summary>

| Columna | Problema | Qué haríamos |
|---|---|---|
| `Fecha` | Dos formatos (`AAAA/MM/DD` y `MM-DD-AA`) | Convertir a `datetime` con el formato correcto |
| `Precio` | Un nulo y un **negativo** (-15) | ¿Devolución o error de digitación? Preguntar al negocio. Imputar o eliminar |
| `Cliente` | Un nulo; puntos inconsistentes ("Maria G" vs "Juan P.") | "Desconocido"; normalizar el texto |
| `Ciudad` | "Bogotá", "Bta" y "bogotá" son la misma ciudad; falta la tilde en "Medellin" | Diccionario de mapeo y minúsculas/mayúsculas consistentes |

Esto es exactamente lo que haremos en el **módulo 04**.
</details>

---
## Ejercicios guiados (15 min)

1. Usa `diccionario` para mostrar la descripción de las 5 variables **más** correlacionadas con `SalePrice`.
2. Verifica la hipótesis del sótano: ¿las casas con todos los `Bsmt*` nulos tienen `TotalBsmtSF == 0`?
3. ¿Hay filas duplicadas si ignoras la columna `Id`? (pista: `df.drop(columns="Id").duplicated().sum()`)
4. Corre `scripts/perfil_inicial.py` sobre `data/raw/housing.csv` y escribe 2 hipótesis para ese dataset.

Soluciones en `ejercicios/soluciones/solucion_03.py`.

In [ ]:
# Tu código aquí